# H3b: Partial Singular Value Equalization — Does Equalizing Only the Top-k SVs Suffice?

## Scientific Context

Muon's orthogonalization step computes the polar factor $UV^T$ of the momentum buffer $M = U\Sigma V^T$, which **sets every singular value to 1**. This is *full spectral democracy*: every direction in the update receives identical weight, regardless of gradient magnitude along that direction.

But a fundamental question remains: **does Muon actually need ALL singular values equalized?** Or is the benefit primarily about suppressing the dominant few?

## The Hypothesis Under Test

Consider the singular value spectrum of a typical momentum buffer: $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_d$. The top singular values carry disproportionate energy. If the optimization benefit comes from preventing *top-SV domination* — where a few large singular values cause the update to collapse onto a low-dimensional subspace — then equalizing only $k=1$ or $k=2$ should recover most of Muon's advantage.

Conversely, if the benefit requires *full democracy* across all directions (including boosting the weakest SVs), then only $k = d$ (full polar) will suffice.

## Protocol

For each $k \in \{1, 2, 4, 8, 16, 32\}$:

1. Compute SVD of momentum: $M = U \Sigma V^T$
2. Set top-$k$ singular values to their mean: $\sigma_1 = \cdots = \sigma_k = \overline{\sigma}_{1:k}$
3. Keep $\sigma_{k+1}, \ldots, \sigma_d$ proportional, rescaled so $\|S_{\text{new}}\|_F = \sqrt{d}$ (matching the polar factor norm)
4. Use the reconstructed matrix as the step direction

LR sweep for each $k$. Measure final loss. Plot loss vs $k$. Find the knee.

## Key Prediction

- If $k=1$ recovers $>50\%$ of Muon's advantage over SGD, then the mechanism is primarily about **top-SV domination suppression**.
- If the curve is gradual (no sharp knee until $k \approx d$), then the mechanism requires **full spectral democracy** across all directions.

## Setup

- 4-layer deep linear network, $32 \times 32$
- 500 training steps, momentum $\beta = 0.9$
- 5 random seeds, batch size 64
- $k \in \{1, 2, 4, 8, 16, 32\}$ (where $k=32$ = full polar = Muon)

In [ ]:
import numpy as np
import os

print("Libraries loaded: numpy", np.__version__)
print("This experiment performs partial SV equalization on the momentum buffer.")
print("We test whether equalizing only the top-k singular values (setting them to")
print("their mean) can recover most of Muon's advantage over plain SGD+momentum.")

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
print(f"Script directory: {SCRIPT_DIR}")

## Hyperparameters

We use a $32 \times 32$ deep linear network with 4 layers. The dimension $d=32$ is chosen so that $k$ can span a meaningful range (1 to 32) on a log-2 scale. 500 training steps are sufficient for the deep linear network to converge under both SGD and Muon, allowing us to measure final-loss differences cleanly.

The momentum coefficient $\beta = 0.9$ ensures the momentum buffer accumulates a non-trivial spectral structure over multiple steps — this is critical because the SV spectrum of the momentum (not the raw gradient) is what we are intervening on.

In [ ]:
DIM = 32
NUM_LAYERS = 4
NUM_STEPS = 500
MOMENTUM = 0.9
NUM_SEEDS = 5
BATCH_SIZE = 64

print("=" * 80)
print("EXPERIMENT CONFIGURATION")
print("=" * 80)
print(f"  Matrix dimension (d):    {DIM}")
print(f"  Number of layers:        {NUM_LAYERS}")
print(f"  Training steps:          {NUM_STEPS}")
print(f"  Momentum coefficient:    {MOMENTUM}")
print(f"  Random seeds:            {NUM_SEEDS}")
print(f"  Batch size:              {BATCH_SIZE}")
print()
print(f"  Polar factor norm:       ||UV^T||_F = sqrt({DIM}) = {np.sqrt(DIM):.4f}")
print(f"  This is the target Frobenius norm for all partial equalization steps.")
print(f"  It ensures that the step magnitude is comparable across all k values,")
print(f"  isolating the DIRECTIONAL effect of equalization from scale effects.")

In [ ]:
K_VALUES = [1, 2, 4, 8, 16, 32]  # 32 = full polar (all SVs equalized)
LR_CANDIDATES = [0.1, 0.07, 0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003, 0.002, 0.001]

print("Equalization sweep values (k):")
for k in K_VALUES:
    frac = k / DIM * 100
    label = " <-- FULL POLAR (Muon)" if k == DIM else ""
    print(f"  k = {k:>3}  ({frac:>5.1f}% of SVs equalized){label}")

print(f"\nLR candidates ({len(LR_CANDIDATES)} values): {LR_CANDIDATES}")
print(f"LR range: [{min(LR_CANDIDATES)}, {max(LR_CANDIDATES)}]")
print()
print("IMPORTANT DESIGN CHOICE: We sweep LR independently for each k.")
print("This is critical because different levels of equalization may have")
print("different optimal step sizes. Without per-k LR tuning, we would")
print("conflate 'wrong LR' with 'equalization hurts performance'.")

## Network Architecture: Deep Linear Model

We use a 4-layer deep linear network $f(X) = W_4 W_3 W_2 W_1 X$ with $32 \times 32$ weight matrices. Despite computing a linear function, deep linear networks exhibit rich optimization dynamics:

- **Loss landscape**: The loss surface has saddle points and poor conditioning that grows with depth.
- **Gradient correlations**: Gradients across layers are coupled through the product structure, creating spectral structure in each layer's momentum buffer.
- **SV spectrum**: The momentum matrices develop anisotropic singular value spectra during training — precisely the structure that Muon's equalization intervenes on.

Weights are initialized as $W_i = I + 0.1 \cdot \mathcal{N}(0, 1)$ (near-identity), so the initial product $\prod W_i \approx I + O(0.1)$, giving a well-conditioned starting point that allows us to measure how optimization *dynamics* (not initialization) drive the results.

In [ ]:
def init_weights(seed):
    rng = np.random.RandomState(seed)
    return [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(NUM_LAYERS)]

# Diagnostic: inspect the spectral structure of one initialization
_demo_weights = init_weights(42)
print("Initial weight matrix diagnostics (seed=42, layer 0):")
_demo_svs = np.linalg.svd(_demo_weights[0], compute_uv=False)
print(f"  Singular values: max={_demo_svs[0]:.4f}, min={_demo_svs[-1]:.4f}")
print(f"  Condition number: {_demo_svs[0]/_demo_svs[-1]:.4f}")
print(f"  Effective rank (||sigma||_1^2 / ||sigma||_2^2): {(np.sum(_demo_svs))**2 / np.sum(_demo_svs**2):.2f} / {DIM}")

_product = _demo_weights[0]
for W in _demo_weights[1:]:
    _product = W @ _product
_prod_svs = np.linalg.svd(_product, compute_uv=False)
print(f"\n  Product W4*W3*W2*W1 singular values:")
print(f"    max={_prod_svs[0]:.4f}, min={_prod_svs[-1]:.4f}, cond={_prod_svs[0]/_prod_svs[-1]:.4f}")
print(f"    The product amplifies the condition number by depth, making optimization harder.")
del _demo_weights, _demo_svs, _product, _prod_svs

In [ ]:
def forward(weights, X):
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

print("forward(weights, X): Computes W_L * ... * W_1 * X sequentially.")
print("  Input shape:  (DIM, BATCH_SIZE) = ({}, {})".format(DIM, BATCH_SIZE))
print("  Output shape: (DIM, BATCH_SIZE) = ({}, {})".format(DIM, BATCH_SIZE))

In [ ]:
def compute_loss(weights, X, Y):
    pred = forward(weights, X)
    return 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))

print("compute_loss: L = (1/2N) * sum_i ||pred_i - y_i||^2")
print("  This is the standard MSE loss (with 1/2 factor for clean gradients).")

In [ ]:
def compute_gradients(weights, X, Y):
    L = len(weights)
    N = X.shape[1]
    acts = [X.copy()]
    for W in weights:
        acts.append(W @ acts[-1])
    delta = (acts[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

print("compute_gradients: Manual backprop through the deep linear network.")
print("  dL/dW_l = delta_l * a_{l-1}^T where delta is backpropagated error.")
print("  Each gradient is a DIM x DIM matrix with its own singular value spectrum.")
print("  The momentum buffer accumulates these over time, building spectral structure.")

## Core Intervention: Partial Singular Value Equalization

This is the heart of the experiment. Given a momentum matrix $M = U \Sigma V^T$ with singular values $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_d$:

**Step 1 — Equalize the top-$k$:** Set $\sigma_1 = \sigma_2 = \cdots = \sigma_k = \bar{\sigma}_{1:k}$ (their mean). This forces the top-$k$ directions to carry equal weight, preventing any single direction from dominating.

**Step 2 — Rescale the rest:** The remaining $\sigma_{k+1}, \ldots, \sigma_d$ are kept *proportional* to their original values but rescaled so the total Frobenius norm equals $\sqrt{d}$ (matching the polar factor). This ensures:
- The total step magnitude is constant across all $k$ values
- The *relative structure* among the bottom $(d-k)$ singular values is preserved
- The only thing that changes is *how many* SVs are equalized

**Boundary cases:**
- $k = 0$: No equalization, just rescale $M$ to norm $\sqrt{d}$ (normalized SGD-like)
- $k = d$: Full equalization = all SVs set to same value = polar factor direction (Muon)

**Why mean rather than 1?** Setting top-$k$ to their mean (rather than 1) before norm-rescaling is a design choice. Since we rescale to a fixed norm afterward, the specific pre-rescale value does not matter for direction — only the *ratio* between equalized and non-equalized SVs matters.

In [ ]:
def partial_sv_equalize(M, k):
    """
    Partial SV equalization of momentum matrix M.

    Given M = U diag(sigma) V^T:
      - Top-k sigmas: set to their mean (equalize them)
      - Remaining sigmas: keep proportional, scale so ||result||_F = ||ortho(M)||_F

    k=DIM gives full polar factor (all SVs = same value => effectively UV^T scaled).
    k=0 gives original M rescaled to match norm.
    """
    U, sigma, Vt = np.linalg.svd(M, full_matrices=False)
    d = len(sigma)
    if np.linalg.norm(sigma) < 1e-15:
        return M

    sigma_new = sigma.copy()

    # Target Frobenius norm: ||ortho(M)||_F = sqrt(d) (polar factor has all SVs=1)
    target_norm = np.sqrt(d)

    # Equalize top-k: set them to their mean
    kk = min(k, d)
    if kk > 0:
        top_mean = np.mean(sigma[:kk])
        sigma_new[:kk] = top_mean

    if kk >= d:
        # Full equalization: all SVs = same value, scale to target norm
        # sigma_new[:] = top_mean => ||sigma_new|| = top_mean * sqrt(d)
        # We want ||result||_F = target_norm = sqrt(d)
        # So scale: sigma_new *= sqrt(d) / (top_mean * sqrt(d)) = 1/top_mean
        if top_mean > 1e-15:
            sigma_new *= target_norm / np.linalg.norm(sigma_new)
        return U @ np.diag(sigma_new) @ Vt

    # Partial equalization: top-k equalized, rest proportional
    # Scale the rest so total ||sigma_new|| = target_norm
    top_energy = np.sum(sigma_new[:kk]**2)
    remaining_energy_budget = target_norm**2 - top_energy

    if remaining_energy_budget > 1e-15:
        rest_current_energy = np.sum(sigma[kk:]**2)
        if rest_current_energy > 1e-15:
            scale = np.sqrt(remaining_energy_budget / rest_current_energy)
            sigma_new[kk:] = sigma[kk:] * scale
        else:
            # Rest is zero, distribute evenly
            sigma_new[kk:] = np.sqrt(remaining_energy_budget / (d - kk))
    else:
        # Top-k already exceeds budget: just normalize to target
        sigma_new[kk:] = 0
        sigma_new *= target_norm / np.linalg.norm(sigma_new)

    return U @ np.diag(sigma_new) @ Vt

# ---- Demonstration: visualize what partial equalization does ----
print("=" * 80)
print("DEMONSTRATION: Partial SV Equalization on a random 32x32 matrix")
print("=" * 80)
_rng = np.random.RandomState(999)
_M_demo = _rng.randn(DIM, DIM)
_svs_original = np.linalg.svd(_M_demo, compute_uv=False)
print(f"\nOriginal M singular values (first 8): {np.round(_svs_original[:8], 3)}")
print(f"  ||M||_F = {np.linalg.norm(_M_demo):.4f}, target = sqrt({DIM}) = {np.sqrt(DIM):.4f}")

for _k in [1, 4, 16, 32]:
    _M_eq = partial_sv_equalize(_M_demo, _k)
    _svs_eq = np.linalg.svd(_M_eq, compute_uv=False)
    print(f"\n  k={_k:>2}: SVs (first 8) = {np.round(_svs_eq[:8], 3)}")
    print(f"        ||result||_F = {np.linalg.norm(_M_eq):.4f}")
    print(f"        Top-{_k} SVs equal? max-min in top-{_k} = {_svs_eq[:_k].max()-_svs_eq[:_k].min():.2e}")

print(f"\nKey observation: as k increases, more SVs become equal.")
print(f"At k={DIM}, ALL SVs are identical = polar factor direction.")
del _rng, _M_demo, _svs_original, _M_eq, _svs_eq, _k

## Training Loops

We define two training functions:

1. **`train_partial_sv`**: The experimental optimizer. At each step:
   - Compute gradient $G_l$ for each layer $l$
   - Update momentum: $m_l \leftarrow \beta \cdot m_l + G_l$
   - Apply partial SV equalization to $m_l$ with parameter $k$
   - Update weights: $W_l \leftarrow W_l - \eta \cdot \text{partial\_sv\_equalize}(m_l, k)$

2. **`train_sgd`**: Plain SGD with momentum (baseline). Same as above but uses the raw momentum buffer directly as the update direction.

The comparison isolates the effect of partial equalization: both methods use the same momentum, same LR tuning procedure, same data, same initialization. The ONLY difference is the spectral intervention applied to the momentum before it becomes the step direction.

In [ ]:
def train_partial_sv(weights_init, X, Y, lr, k):
    """Train with partial SV equalization applied to momentum."""
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]

    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y)
        if not np.isfinite(loss) or loss > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y)
        for i in range(NUM_LAYERS):
            mom[i] = MOMENTUM * mom[i] + grads[i]
            step_dir = partial_sv_equalize(mom[i], k)
            weights[i] = weights[i] - lr * step_dir
    return compute_loss(weights, X, Y)

print("train_partial_sv: Optimizer with partial SV equalization on momentum.")
print(f"  At each step, the momentum buffer's top-k SVs are set to their mean,")
print(f"  then the whole spectrum is rescaled to ||step||_F = sqrt({DIM}).")
print(f"  This is applied independently to each of the {NUM_LAYERS} layers.")

In [ ]:
def train_sgd(weights_init, X, Y, lr):
    """Plain SGD with momentum (baseline)."""
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros((DIM, DIM)) for _ in range(NUM_LAYERS)]

    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y)
        if not np.isfinite(loss) or loss > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y)
        for i in range(NUM_LAYERS):
            mom[i] = MOMENTUM * mom[i] + grads[i]
            weights[i] = weights[i] - lr * mom[i]
    return compute_loss(weights, X, Y)

print("train_sgd: Plain SGD with momentum=0.9 (baseline).")
print("  No spectral intervention. The raw momentum buffer is used as the step.")
print("  This baseline has anisotropic updates — large SVs dominate the step.")

In [ ]:
def make_data(seed):
    rng = np.random.RandomState(seed)
    W_target = rng.randn(DIM, DIM) * 0.5
    X = rng.randn(DIM, BATCH_SIZE) * 0.3
    Y = W_target @ X
    return X, Y

# Diagnostic: inspect the target and data properties
_X_demo, _Y_demo = make_data(42)
_W_target_demo = np.random.RandomState(42).randn(DIM, DIM) * 0.5
_target_svs = np.linalg.svd(_W_target_demo, compute_uv=False)
print("Data generation diagnostics (seed=42):")
print(f"  Target matrix W*: {DIM}x{DIM}, entries ~ N(0, 0.5)")
print(f"  Target SV range: [{_target_svs[-1]:.3f}, {_target_svs[0]:.3f}]")
print(f"  Target condition number: {_target_svs[0]/_target_svs[-1]:.2f}")
print(f"  Input X shape: {_X_demo.shape}, entries ~ N(0, 0.3)")
print(f"  Output Y = W* @ X shape: {_Y_demo.shape}")
print(f"  Initial loss (random init): {compute_loss(init_weights(42+5000), _X_demo, _Y_demo):.4f}")
del _X_demo, _Y_demo, _W_target_demo, _target_svs

## Learning Rate Sweep

A critical methodological detail: **each value of $k$ gets its own independently-tuned learning rate.** This prevents a common confound in optimizer comparison studies where one method appears worse simply because it was given a suboptimal LR.

The sweep uses the first 3 seeds for LR selection (to avoid overfitting the LR to all seeds), then evaluates with all 5 seeds at the selected LR. This is a standard train/validation split applied to hyperparameter selection.

In [ ]:
def sweep_lr(train_fn, seeds, candidates):
    """
    Sweep over LR candidates, use first 3 seeds for selection.
    train_fn(weights_init, X, Y, lr) -> final_loss.
    """
    best_lr, best_loss = candidates[-1], float('inf')
    for lr in candidates:
        losses = []
        for s in seeds[:3]:
            X, Y = make_data(s)
            w = init_weights(s + 5000)
            fl = train_fn(w, X, Y, lr)
            losses.append(fl)
        finite = [l for l in losses if np.isfinite(l)]
        ml = np.mean(finite) if finite else float('inf')
        if ml < best_loss:
            best_loss = ml
            best_lr = lr
    return best_lr

print("sweep_lr: Evaluates each LR candidate on 3 seeds, picks the one with lowest mean loss.")
print(f"  {len(LR_CANDIDATES)} candidates x 3 seeds = {len(LR_CANDIDATES)*3} training runs per k value.")
print(f"  Total LR sweep budget: {len(LR_CANDIDATES)*3*(len(K_VALUES)+1)} runs (for {len(K_VALUES)} k values + SGD).")

## Main Experiment Execution

The experiment proceeds in two phases:

**Phase 1 — LR Selection:** For each $k$ value (and SGD baseline), sweep over 12 learning rate candidates using 3 seeds. Select the LR that minimizes mean final loss. This ensures each method is evaluated at its best operating point.

**Phase 2 — Full Evaluation:** Train with all 5 seeds at the selected LR. Report mean and standard deviation of final loss.

**Analysis:** For each $k$, compute:
- **Loss**: How well does partial equalization with this $k$ perform?
- **% Gap Closed**: What fraction of the SGD-to-Muon performance gap does this $k$ capture? $(L_{\text{SGD}} - L_k) / (L_{\text{SGD}} - L_{\text{Muon}}) \times 100\%$
- **Marginal Gain**: How much additional gap closure does increasing $k$ to this level provide beyond the previous $k$?

The gap closure metric is the key output: it tells us exactly how much of Muon's benefit can be attributed to equalizing the top-$k$ singular values versus the remaining ones.

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print("=" * 100)
    print("H3b: PARTIAL SV EQUALIZATION -- Does Equalizing Only Top-k SVs Suffice?")
    print("=" * 100)
    print(f"k values: {K_VALUES} (k={DIM} = full polar)")
    print(f"Network: {NUM_LAYERS}-layer deep linear, {DIM}x{DIM}, {NUM_STEPS} steps, {NUM_SEEDS} seeds")
    print(f"Seeds: {seeds}")
    print()

    # ========================================================================
    # Phase 1: LR sweep for each k
    # ========================================================================
    print("=" * 100)
    print("PHASE 1: Learning Rate Selection")
    print("=" * 100)
    print(f"Using first 3 seeds for LR selection: {seeds[:3]}")
    print(f"Sweeping {len(LR_CANDIDATES)} LR candidates per optimizer configuration.\n")

    best_lrs = {}
    for k in K_VALUES:
        fn = lambda w, X, Y, lr, _k=k: train_partial_sv(w, X, Y, lr, _k)
        best_lr = sweep_lr(fn, seeds, LR_CANDIDATES)
        best_lrs[k] = best_lr
        frac = k / DIM * 100
        print(f"  k={k:>3} ({frac:>5.1f}% equalized): best_lr={best_lr:.4f}")

    # Also sweep SGD baseline
    sgd_lr = sweep_lr(train_sgd, seeds, LR_CANDIDATES)
    print(f"  SGD  (0.0% equalized):  best_lr={sgd_lr:.4f}")

    print(f"\n  LR trend observation:")
    print(f"    SGD best LR:       {sgd_lr:.4f}")
    print(f"    k=1 best LR:       {best_lrs[1]:.4f}")
    print(f"    k={DIM} (Muon) LR:  {best_lrs[DIM]:.4f}")
    if best_lrs[DIM] > sgd_lr:
        print(f"    --> Full equalization tolerates HIGHER LR than SGD (ratio: {best_lrs[DIM]/sgd_lr:.1f}x)")
        print(f"       This is expected: norm-controlled updates are more stable.")
    elif best_lrs[DIM] < sgd_lr:
        print(f"    --> Full equalization uses LOWER LR than SGD (ratio: {best_lrs[DIM]/sgd_lr:.2f}x)")
    else:
        print(f"    --> Same optimal LR for both methods.")

    # ========================================================================
    # Phase 2: Full training with all seeds
    # ========================================================================
    print(f"\n{'=' * 100}")
    print("PHASE 2: Full Training with All Seeds")
    print("=" * 100)
    print(f"Evaluating each k with all {NUM_SEEDS} seeds at its best LR.\n")

    results_k = {}
    for k in K_VALUES:
        losses = []
        for s in seeds:
            X, Y = make_data(s)
            w = init_weights(s + 5000)
            fl = train_partial_sv(w, X, Y, best_lrs[k], k)
            losses.append(fl)
        finite = [l for l in losses if np.isfinite(l)]
        mean_loss = np.mean(finite) if finite else float('inf')
        std_loss = np.std(finite) if len(finite) > 1 else 0.0
        results_k[k] = {'mean': mean_loss, 'std': std_loss, 'lr': best_lrs[k]}
        n_finite = len(finite)
        n_diverged = NUM_SEEDS - n_finite
        status = "" if n_diverged == 0 else f"  [WARNING: {n_diverged}/{NUM_SEEDS} seeds diverged!]"
        print(f"  k={k:>3}: loss={mean_loss:.6e} +/- {std_loss:.6e}  (lr={best_lrs[k]:.4f}, {n_finite}/{NUM_SEEDS} converged){status}")

    # SGD baseline
    sgd_losses = []
    for s in seeds:
        X, Y = make_data(s)
        w = init_weights(s + 5000)
        fl = train_sgd(w, X, Y, sgd_lr)
        sgd_losses.append(fl)
    sgd_finite = [l for l in sgd_losses if np.isfinite(l)]
    sgd_mean = np.mean(sgd_finite) if sgd_finite else float('inf')
    sgd_std = np.std(sgd_finite) if len(sgd_finite) > 1 else 0.0
    print(f"  SGD:   loss={sgd_mean:.6e} +/- {sgd_std:.6e}  (lr={sgd_lr:.4f})")

    # ==========================================================================
    # RESULTS TABLE
    # ==========================================================================

    muon_loss = results_k[DIM]['mean']  # k=32 = full polar = Muon
    total_gap = sgd_mean - muon_loss

    print(f"\n\n{'=' * 100}")
    print("RESULTS: Final Loss vs Number of Equalized SVs (k)")
    print(f"{'=' * 100}")
    print(f"\n  SGD baseline loss:  {sgd_mean:.6e}  (no equalization, raw momentum)")
    print(f"  Full Muon loss:     {muon_loss:.6e}  (all {DIM} SVs equalized)")
    print(f"  Total gap:          {total_gap:.6e}  (this is the performance we are trying to explain)")
    print(f"  Ratio SGD/Muon:     {sgd_mean/max(muon_loss, 1e-30):.2f}x")

    print(f"\n  {'k':>5}  {'Loss':>14}  {'Std':>14}  {'LR':>8}  {'vs SGD':>10}  {'vs Muon':>10}  {'% Gap Closed':>14}")
    print("  " + "-" * 82)
    print(f"  {'SGD':>5}  {sgd_mean:>14.6e}  {sgd_std:>14.6e}  {sgd_lr:>8.4f}  {'ref':>10}  {sgd_mean/max(muon_loss,1e-30):>10.2f}x  {'0.0%':>14}")
    for k in K_VALUES:
        r = results_k[k]
        vs_sgd = sgd_mean / max(r['mean'], 1e-30)
        vs_muon = r['mean'] / max(muon_loss, 1e-30)
        gap_closed = (sgd_mean - r['mean']) / max(total_gap, 1e-30) * 100 if total_gap > 1e-30 else 0
        marker = "  <-- full polar (Muon)" if k == DIM else ""
        print(f"  {k:>5}  {r['mean']:>14.6e}  {r['std']:>14.6e}  {r['lr']:>8.4f}  "
              f"{vs_sgd:>10.2f}x  {vs_muon:>10.2f}x  {gap_closed:>13.1f}%{marker}")

    # ==========================================================================
    # INTERPRETATION: READING THE TABLE
    # ==========================================================================

    print(f"\n  --- Reading the table ---")
    print(f"  'vs SGD' = how many times better than SGD (higher = better optimizer)")
    print(f"  'vs Muon' = ratio to full Muon loss (1.00 = matches Muon, >1 = worse)")
    print(f"  '% Gap Closed' = fraction of SGD-Muon gap captured (100% = full Muon)")

    # ==========================================================================
    # HYPOTHESIS TESTS
    # ==========================================================================

    print(f"\n\n{'=' * 100}")
    print("HYPOTHESIS TESTS")
    print(f"{'=' * 100}")

    # T1: k=1 recovers >50% of full polar advantage
    gap_k1 = (sgd_mean - results_k[1]['mean']) / max(total_gap, 1e-30) * 100 if total_gap > 1e-30 else 0
    t1 = gap_k1 > 50
    print(f"\n  T1: k=1 (suppress only top SV) captures >50% of Muon advantage?")
    print(f"      ---------------------------------------------------------------")
    print(f"      SGD loss:  {sgd_mean:.6e}")
    print(f"      k=1 loss:  {results_k[1]['mean']:.6e}")
    print(f"      Muon loss: {muon_loss:.6e}")
    print(f"      Gap closed by k=1: {gap_k1:.1f}%")
    print(f"      Threshold: 50%")
    if t1:
        print(f"      --> PASS: Mechanism is primarily about top-SV domination suppression.")
        print(f"         Equalizing just the SINGLE largest singular value of the momentum")
        print(f"         buffer captures more than half of Muon's benefit. This means the")
        print(f"         dominant failure mode of SGD is that one direction hogs the update.")
    else:
        print(f"      --> FAIL: Need more than top-1 equalization; bottom SVs matter too.")
        print(f"         The benefit of Muon is NOT primarily about suppressing the top SV.")
        print(f"         Instead, it requires redistributing energy across MANY directions.")
        print(f"         This suggests Muon's value comes from boosting weak directions,")
        print(f"         not (just) from suppressing the dominant one.")

    # T2: Find the knee (smallest k capturing >80% of gap)
    knee_k = None
    for k in K_VALUES:
        gap_k = (sgd_mean - results_k[k]['mean']) / max(total_gap, 1e-30) * 100 if total_gap > 1e-30 else 0
        if gap_k > 80:
            knee_k = k
            break
    t2 = knee_k is not None and knee_k < DIM // 2
    print(f"\n  T2: Sharp knee at k << dim (>80% gap captured at k < {DIM//2})?")
    print(f"      ---------------------------------------------------------------")
    if knee_k is not None:
        gap_at_knee = (sgd_mean - results_k[knee_k]['mean']) / max(total_gap, 1e-30) * 100
        print(f"      Knee at k={knee_k} ({gap_at_knee:.1f}% gap closed)")
        print(f"      Threshold: k < {DIM//2} (less than half the dimensions)")
        if t2:
            print(f"      --> PASS: Sharp knee exists at k={knee_k}, well below d/2={DIM//2}.")
            print(f"         Only {knee_k}/{DIM} SVs ({knee_k/DIM*100:.0f}% of spectrum) need equalization")
            print(f"         to recover >80% of Muon's benefit. The top few SVs dominate")
            print(f"         the momentum spectrum, and equalizing them is (nearly) sufficient.")
        else:
            print(f"      --> FAIL: Knee at k={knee_k} >= d/2={DIM//2}.")
            print(f"         Need to equalize at least half the SVs for >80% benefit.")
            print(f"         The benefit is distributed across the spectrum, not concentrated")
            print(f"         in the top few singular values.")
    else:
        print(f"      No knee found (no k achieves >80% gap closure)")
        print(f"      --> FAIL: Even full equalization does not close >80% of the gap,")
        print(f"         or the gap closure is very gradual across k values.")

    # T3: k=DIM exactly recovers Muon
    # Since k=DIM IS our Muon proxy, this is a sanity check (should be ~100%)
    gap_full = (sgd_mean - results_k[DIM]['mean']) / max(total_gap, 1e-30) * 100 if total_gap > 1e-30 else 0
    t3 = gap_full > 95
    print(f"\n  T3: k={DIM} (full equalization) recovers full polar performance?")
    print(f"      ---------------------------------------------------------------")
    print(f"      k={DIM} gap closed: {gap_full:.1f}%")
    print(f"      Threshold: >95% (sanity check)")
    if t3:
        print(f"      --> PASS (sanity check): k={DIM} = full equalization correctly")
        print(f"         recovers Muon-level performance. The implementation is consistent.")
    else:
        print(f"      --> FAIL (implementation issue?): k={DIM} does not match Muon.")
        print(f"         This should be exactly 100% by construction. If it is not,")
        print(f"         there may be a numerical issue in the equalization procedure.")

    # ==========================================================================
    # DIMINISHING RETURNS ANALYSIS
    # ==========================================================================

    print(f"\n\n{'=' * 100}")
    print("DIMINISHING RETURNS ANALYSIS")
    print(f"{'=' * 100}")
    print(f"\n  This section examines the MARGINAL contribution of each doubling of k.")
    print(f"  If the curve shows strong diminishing returns (most gain from k=1),")
    print(f"  the mechanism is 'top-heavy'. If gains are evenly distributed,")
    print(f"  the mechanism requires 'full democracy'.\n")
    print(f"  Marginal gain from increasing k:")
    prev_gap = 0
    for k in K_VALUES:
        gap_k = (sgd_mean - results_k[k]['mean']) / max(total_gap, 1e-30) * 100 if total_gap > 1e-30 else 0
        marginal = gap_k - prev_gap
        bar = "#" * int(max(marginal, 0) / 2)
        print(f"    k={k:>3}: {gap_k:>7.1f}% total  (marginal +{marginal:>5.1f}%)  {bar}")
        prev_gap = gap_k

    # Compute where the 50% point of marginal gains lies
    gaps_by_k = {}
    for k in K_VALUES:
        gaps_by_k[k] = (sgd_mean - results_k[k]['mean']) / max(total_gap, 1e-30) * 100 if total_gap > 1e-30 else 0
    total_marginal_sum = gaps_by_k[K_VALUES[-1]]  # should be ~100%
    cumulative = 0
    half_k = K_VALUES[-1]
    for k in K_VALUES:
        marginal = gaps_by_k[k] - (gaps_by_k[K_VALUES[K_VALUES.index(k)-1]] if K_VALUES.index(k) > 0 else 0)
        cumulative += marginal
        if cumulative >= total_marginal_sum / 2:
            half_k = k
            break
    print(f"\n  50% of total marginal gain is achieved by k={half_k}.")
    if half_k <= K_VALUES[1]:
        print(f"  --> Strongly top-heavy: the first 1-2 SVs account for most of the benefit.")
    elif half_k <= K_VALUES[len(K_VALUES)//2]:
        print(f"  --> Moderately concentrated: top ~{half_k} SVs carry most weight.")
    else:
        print(f"  --> Distributed: gains are spread across many SVs.")

    # ==========================================================================
    # CONCLUSION
    # ==========================================================================

    print(f"\n\n{'=' * 100}")
    print("CONCLUSION")
    print(f"{'=' * 100}")

    tests_passed = sum([t1, t2, t3])
    print(f"\n  Tests passed: {tests_passed}/3")

    if t1:
        print(f"\n  PRIMARY FINDING: Equalizing just the top SV captures >{gap_k1:.0f}% of Muon's")
        print(f"  advantage. The mechanism is primarily about preventing top-SV domination.")
        print(f"  This supports the 'spectral democracy' interpretation: the main benefit")
        print(f"  comes from preventing any single direction from dominating the update.")
    else:
        print(f"\n  PRIMARY FINDING: Top-1 equalization captures only {gap_k1:.0f}% of the gap.")
        print(f"  Muon's benefit requires equalizing MANY singular values, not just")
        print(f"  suppressing the dominant one. This supports the 'full democracy'")
        print(f"  interpretation: ALL directions need equal voice.")

    if knee_k is not None:
        print(f"\n  KNEE: k={knee_k} captures >80% of the advantage.")
        if knee_k <= DIM // 4:
            print(f"  Only {knee_k}/{DIM} SVs need equalization -- the top few dominate.")
        else:
            print(f"  Need {knee_k}/{DIM} SVs equalized -- benefit is distributed.")
    else:
        print(f"\n  NO KNEE: Need full equalization for >80% benefit.")

    print(f"\n  IMPLICATIONS FOR MUON THEORY:")
    if t1 and t2:
        print(f"  The partial equalization results strongly suggest that Muon's benefit")
        print(f"  is primarily a 'top-SV suppression' effect. A lightweight approximation")
        print(f"  that only equalizes the top few SVs could achieve most of Muon's benefit")
        print(f"  at lower computational cost (no need for full SVD, just a few power iterations).")
    elif t1 and not t2:
        print(f"  Top-1 equalization captures most of the benefit, but reaching >80% requires")
        print(f"  equalizing many more SVs. This suggests a two-factor mechanism:")
        print(f"  (1) Top-SV suppression provides the primary benefit.")
        print(f"  (2) Bottom-SV boosting provides a secondary but non-negligible boost.")
    else:
        print(f"  Full spectral democracy is required for Muon's benefit. Partial")
        print(f"  equalization is insufficient — the optimizer genuinely needs ALL")
        print(f"  directions to carry equal weight in the update step. This argues")
        print(f"  against cheap approximations that only handle the top few SVs.")

    print(f"\n{'=' * 100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'=' * 100}")

    # ==========================================================================
    # PLOT
    # ==========================================================================
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        fig.suptitle(f'H3b: Partial SV Equalization\n'
                     f'{NUM_LAYERS}-layer deep linear {DIM}x{DIM}, {NUM_STEPS} steps, {NUM_SEEDS} seeds',
                     fontsize=13, fontweight='bold')

        # (a) Loss vs k
        ax = axes[0]
        k_vals = K_VALUES
        loss_vals = [results_k[k]['mean'] for k in k_vals]
        std_vals = [results_k[k]['std'] for k in k_vals]
        ax.errorbar(k_vals, loss_vals, yerr=std_vals, marker='o', linewidth=2,
                     capsize=4, color='#CC3311', label='Partial SV Eq.')
        ax.axhline(y=sgd_mean, color='#4477AA', linestyle='--', linewidth=1.5, label=f'SGD ({sgd_mean:.2e})')
        ax.axhline(y=muon_loss, color='#228B22', linestyle='--', linewidth=1.5, label=f'Full Polar ({muon_loss:.2e})')
        ax.set_xlabel('k (number of SVs equalized)')
        ax.set_ylabel('Final Loss')
        ax.set_yscale('log')
        ax.set_xscale('log', base=2)
        ax.set_xticks(k_vals)
        ax.set_xticklabels([str(k) for k in k_vals])
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_title('(a) Loss vs k')

        # (b) % Gap Closed vs k
        ax = axes[1]
        gaps = [(sgd_mean - results_k[k]['mean']) / max(total_gap, 1e-30) * 100 for k in k_vals]
        ax.plot(k_vals, gaps, marker='s', linewidth=2, color='#9933CC')
        ax.axhline(y=50, color='gray', linestyle=':', alpha=0.5, label='50% threshold (T1)')
        ax.axhline(y=80, color='gray', linestyle='--', alpha=0.5, label='80% threshold (T2)')
        ax.axhline(y=100, color='#228B22', linestyle='-', alpha=0.3, label='100% (full polar)')
        ax.set_xlabel('k (number of SVs equalized)')
        ax.set_ylabel('% of SGD-Muon Gap Closed')
        ax.set_xscale('log', base=2)
        ax.set_xticks(k_vals)
        ax.set_xticklabels([str(k) for k in k_vals])
        ax.set_ylim(-5, 110)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_title('(b) Gap Closure vs k')

        # (c) Marginal gain
        ax = axes[2]
        marginals = [gaps[0]] + [gaps[i] - gaps[i-1] for i in range(1, len(gaps))]
        ax.bar(range(len(k_vals)), marginals, color='#FF8800', edgecolor='black')
        ax.set_xticks(range(len(k_vals)))
        ax.set_xticklabels([f'k={k}' for k in k_vals], rotation=30)
        ax.set_ylabel('Marginal % Gap Closed')
        ax.set_title('(c) Marginal Gain per k Step')
        ax.grid(True, alpha=0.3, axis='y')
        for i, v in enumerate(marginals):
            ax.text(i, max(v, 0) + 0.5, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)

        plt.tight_layout()
        plot_path = os.path.join(SCRIPT_DIR, 'h3b_partial_sv_equalization.png')
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"\nPlot saved: {plot_path}")
    except ImportError:
        print("\nWARNING: matplotlib not available, skipping plot.")

## Execution

Running the full experiment below. Expected runtime: ~2-5 minutes depending on hardware (the LR sweep dominates the cost, with ~252 training runs total).

In [ ]:
if __name__ == '__main__':
    main()

## Interpretation Guide

### How to read the results

The three plots produced above answer the core question from three complementary angles:

**(a) Loss vs k** — Shows absolute performance of each partial equalization level. The curve should be monotonically decreasing (more equalization = better performance) if the "spectral democracy" hypothesis is correct. Watch for:
- Does k=1 already drop significantly below SGD? (Top-SV suppression matters)
- Is the curve convex or concave? (Convex = diminishing returns; concave = accelerating returns)

**(b) Gap Closure vs k** — The key diagnostic plot. This normalizes away the absolute loss scale and focuses on what fraction of the SGD-to-Muon gap is closed by each k. The shape of this curve tells the whole story:
- **Step function** (jumps to ~100% at k=1): Mechanism is purely top-SV suppression
- **Logarithmic/concave** (fast rise, then plateau): Mechanism is top-heavy, a few SVs dominate
- **Linear** (steady climb): Mechanism is uniformly distributed across the spectrum
- **Convex/exponential** (slow start, late acceleration): Mechanism is bottom-heavy, weak SVs matter most

**(c) Marginal Gain** — Shows the *incremental* contribution of each doubling of k. If the first bar (k=1) is the tallest, the mechanism is top-heavy. If the last bar (k=32) is the tallest, full equalization is essential and partial approaches lose most of the benefit.

### Connection to broader theory

This experiment directly tests the "spectral democracy budget" concept: how much equalization does Muon need? The answer has practical implications:
- If partial equalization suffices, cheaper approximations (e.g., power iteration to find top few SVs) could replace full SVD
- If full equalization is needed, the polar factor is not just a convenient normalization but an essential component of the algorithm

### Caveats

- Deep linear networks may not capture all the dynamics of nonlinear networks (e.g., activation-dependent curvature)
- The $32 \times 32$ dimension is small; real networks have much larger weight matrices where the SV spectrum may behave differently
- The "equalize top-k to their mean" intervention conflates two effects: (1) suppressing large SVs and (2) boosting small ones. A separate experiment that only clamps (without boosting) would disambiguate these